# 03-01_SCPダウンロード
## 目的
- Telnet接続した踏み台サーバから、JupyterHub上にSCPでダウンロードする
- ダウンロード対象のファイルは他サーバーから踏み台サーバにダウンロードして集約する

## 現在のファイル転送ルート
- ターゲットサーバー -> 中継サーバー -> 踏み台サーバ -> Jupyterhub
- ファイル転送に使うディレクトリはログインユーザーのホームディレクトリを指定する

# モジュールのインポート

In [ ]:
# ==========================================
# 1. ライブラリのインポート
# ==========================================
# 標準ライブラリ・外部ライブラリ
import sys
import os
import json
import re
import pexpect
import time
from datetime import datetime
from netmiko import ConnectHandler
from getpass import getpass

# 自作モジュール（ネットワーク接続・実行管理用）
from module.login_utils import login_telnet, login_jump_ssh, enable_mode

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: 必要なモジュールの読み込みが完了しました。")
print("="*50)

# ユーザー事前定義

In [ ]:
# ==========================================
# 2. 実行対象と設定ファイルの定義
# ==========================================

# 接続対象の指定
jump_server_key = "clab-shuttle01"
ssh1_server_key = "sandbox"
ssh2_server_key = "pve-nuc01"

# 入力ファイルパスの指定
device_file = "data/device_info.json" # デバイス接続情報

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: パラメータ定義をロードしました。")
print(f"  - サーバ　　: {jump_server_key}")
print(f"  - サーバ　　: {ssh1_server_key}")
print(f"  - サーバ　　: {ssh2_server_key}")
print(f"  - 接続情報　: {device_file}")
print("="*50)

# デバイス情報

In [ ]:
# ==========================================
# 3. 接続情報の展開と実行準備
# ==========================================

# --- 構成情報の読み込み ---
# JSONから指定したホストの接続パラメータ（IP、ポート、OS種別等）を抽出
with open(device_file, 'r') as f:
    device_info = json.load(f)

# サーバーのプロファイル
device_jump = device_info[jump_server_key]    
device_ssh1 = device_info[ssh1_server_key]
device_ssh2 = device_info[ssh2_server_key]

# --- 認証情報の補完 ---
# JSONにパスワードが未記載の場合、インタラクティブに入力を促す
if not device_jump.get('password'):
    device_jump['password'] = getpass(f"Enter サーバー({jump_server_key}) のパスワード: ")

if not device_jump.get('password'):
    device_ssh1['password'] = getpass(f"Enter サーバー{ssh1_server_key}) のパスワード: ")

if not device_jump.get('password'):
    device_ssh2['password'] = getpass(f"Enter サーバー({ssh2_server_key}) のパスワード: ")

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [INFO]: 準備が整いました。")
print(f"  - 接続ルート　: {jump_server_key}({device_jump['host']}) ->  {ssh1_server_key}({device_ssh1['host']}) ->  {ssh2_server_key}({device_ssh2['host']})")
print("="*50)
print("⚠️  [CHECK]: 接続ルートが意図したものか、上記を確認してください。")

# ログイン

In [ ]:
# ==========================================
# 4. サーバーへの接続実行
# ==========================================

# --- 踏み台サーバへのログイン ---
# Telnetを使用して中継地点(Jump Server)に接続
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: 踏み台サーバ ({jump_server_key}) へTelnet接続中...")
conn = login_telnet(device_jump, jump_server_key, log_file="output/scp-transfer.log")

# --- ターゲットデバイスへのログイン ---
# 踏み台のセッションを維持したまま、SSHで最終目的地へ接続
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: ターゲット ({ssh1_server_key}) へSSH接続を試行します...")
conn = login_jump_ssh(
    conn, 
    device_ssh1["host"], 
    device_ssh1["username"], 
    device_ssh1["password"], 
    label=ssh1_server_key
)

print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: ターゲット ({ssh2_server_key}) へSSH接続を試行します...")
conn = login_jump_ssh(
    conn, 
    device_ssh2["host"], 
    device_ssh2["username"], 
    device_ssh2["password"], 
    label=ssh2_server_key
)

# --- 接続完了確認 ---
print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [SUCCESS]: ログイン処理が完了しました。")
print(f"現在のプロンプト: {conn.find_prompt()}")
print("="*50)
print("⚠️  [CHECK]: プロンプトが意図した機器のものか、上記を確認してください。")

# SCPファイルダウンロード

In [ ]:
# ==========================================
# 5. ファイルの確認とSCP転送（クリーンアップ機能付き）
# ==========================================

# --- 保存先ディレクトリの作成 (ローカル) ---
os.makedirs("output", exist_ok=True)

# --- ssh2_serverでファイル一覧を確認 ---
print(f"[{datetime.now().strftime('%H:%M:%S')}] [RUN]: {ssh2_server_key} のファイル一覧を確認します...")
files_list = conn.send_command("ls -l")
print(files_list)

# --- ダウンロード対象ファイルの手動入力 ---
target_file = input("\nダウンロードするファイル名を入力してください: ")

# --- サーバー間SCPの実行 ---
ssh_option = "-q -l 1024 -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null"

# ssh2_server_key -> ssh1_server_keyへSCPアップロード
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] [STEP 1]: {ssh2_server_key} -> {ssh1_server_key} へ転送中...")
scp_cmd1 = f"scp {ssh_option} {target_file} {device_ssh1['username']}@{device_ssh1['host']}:/home/{device_ssh1['username']}"
output = conn.send_command_timing(scp_cmd1, strip_prompt=False)
if "password" in output.lower():
    output = conn.send_command_timing(device_ssh1['password'])
if f"{device_ssh2['username']}" in output:
    conn.send_command("exit")

# ssh1_server_key -> jump_server_keyへSCPアップロード
print(f"[{datetime.now().strftime('%H:%M:%S')}] [STEP 2]: {ssh1_server_key} -> {jump_server_key} へ転送中...")
scp_cmd2 = f"scp {ssh_option} {target_file} {device_jump['username']}@{device_jump['host']}:/home/{device_jump['username']}"
output = conn.send_command_timing(scp_cmd2, strip_prompt=False)
if "password" in output.lower():
    output = conn.send_command_timing(device_jump['password'])
output = conn.send_command_timing(device_jump['password'])
if f"{device_ssh1['username']}" in output:
    conn.send_command("exit")

# --- ローカル(output/)へSCPダウンロード ---
print(f"[{datetime.now().strftime('%H:%M:%S')}] [STEP 3]: {jump_server_key} -> ローカル(output/) へダウンロード中...")

# 踏み台から完全にログアウトしてローカルJupyterHubに戻る
# ※切断前に一時ファイルの削除予約が必要なため、一度シェルを抜ける前に管理情報を取得
jump_host = device_jump['host']
jump_user = device_jump['username']
jump_pwd = device_jump['password']

# JupyterHub(ローカル)のOSコマンドでSCP実行
local_scp_cmd = f"scp {ssh_option} {jump_user}@{jump_host}:/home/{device_jump['username']}/{target_file} ./output/"
child = pexpect.spawn(local_scp_cmd)
child.expect("assword:")
child.sendline(jump_pwd)
child.wait()

print("="*50)
print(f"[{datetime.now().strftime('%H:%M:%S')}] [SUCCESS]: 全工程が完了しました。")
print("="*50)